[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/halla-ai/deepnlp-2026/blob/main/notebooks/week-10.ipynb)

# 10주차 실습: 정렬 목적함수와 GRPO 보상 broadcast 관찰

- 목표: SFT·선호·PPO·DPO의 준비된 목적함수 예제를 비교한 뒤 outcome GRPO의 숫자 관찰
- 환경: Python 3, CPU, 표준 라이브러리만 사용
- 코드 변경 과제 없음. 모든 셀을 순서대로 실행
- 실제 모델 학습, rollout, tokenizer, verifier 실행 없음
- reward와 길이는 교수 제공 산술 예시. 출력은 학습 성능 측정값이 아님
- 수업계획서와 이슈 #4에 따른 관찰 전용 실습

## 출처와 공개 정보 맥락

- GRPO: [DeepSeekMath 4.1.2](https://arxiv.org/html/2402.03300v3#S4.SS1.SSS2)
- 집계: [DAPO 3.3](https://arxiv.org/html/2503.14476v1#S3.SS3)
- 제주 공개 정보: [위키백과 「제주특별자치도」 고정판 42464087](https://ko.wikipedia.org/w/index.php?title=제주특별자치도&oldid=42464087), 7주차와 동일 출처
- 질문 "제주특별자치도의 두 행정시는?": 제주시·서귀포시의 정적 정보와 최신 운영정보의 차이 검토
- 원문 데이터 미포함. 관광 응대 후보는 교수 작성 예시, 실제 기업 데이터나 모델 생성 결과 아님


## 0. 준비

- Python 3 CPU 환경, 추가 패키지 설치·모델 다운로드 불필요
- 전체 셀을 수정 없이 순서대로 실행. 직접 구현이나 학습 요구 없음
- 아래 제공값은 모두 교육용 산술 예제. 실제 모델의 정책 확률·보상·성능 측정값 아님
- 한 상정 질문의 감독 입력이 SFT 시연, 선호 라벨, 정책 로그비, 그룹 보상으로 어떻게 달라지는지 먼저 예상
- 정답 공개 여부·점수와 평가자의 판단·실제 응답 안전성을 혼동하지 않기


## 1. 제공 예제 실행

### 1-1. 한 질문의 시연과 선호를 숫자로 읽기

- 공통 질문: 제주 관광지의 최신 운영 공지를 확인하지 못했는데 내일 개방한다고 확답해도 되는가?
- U: "내일 정상 운영합니다. 바로 방문하세요."
- V: "최신 운영 공지가 없어 확답하기 어렵습니다. 공식 운영 안내를 확인한 뒤 방문 일정을 정하세요."
- 교수 작성 상정 질문과 예시 응답, 실제 관광지 운영 사실이나 모델 출력 아님
- 목표 V의 세 위치 [확인 / 후 / 방문]에 확률0.5,0.25,0.8 제공. 실제 tokenizer 결과 아님
- SFT teacher forcing은 정답 접두어를 다음 위치 입력으로 사용. 선호 예제는 V가 U보다 낫다는 비교 라벨 사용
- 코드를 바꾸지 않고 토큰 손실과 선호확률이 무엇을 평가하는지 관찰


In [ ]:
import math

sft_probabilities = [0.5, 0.25, 0.8]
sft_losses = [-math.log(p) for p in sft_probabilities]
sft_ce = sum(sft_losses) / len(sft_losses)
print("SFT 제공 토큰 확률:", sft_probabilities)
print("토큰 CE:", [round(loss, 6) for loss in sft_losses])
print("평균 CE:", round(sft_ce, 6))
print("모두0.8인 preset CE:", round(-math.log(0.8), 6))

preference_examples = []
for gap in [0.0, 2.0, -2.0]:
    probability = 1 / (1 + math.exp(-gap))
    loss = -math.log(probability)
    preference_examples.append({"reward_gap": gap, "probability": probability, "loss": loss})
    print("보상점수차", gap, "V선호 확률", round(probability, 6), "V선호 라벨 손실", round(loss, 6))
print("선호확률은 사실 정확도나 안전 확률이 아님")


### 1-2. PPO clip의 네 가지 경우

- old token probability0.20, new0.30 또는0.10, epsilon0.2 제공
- A=+1 또는-1의 네 경우를 같은 함수식으로 실행
- 출력 J는 최대화하는 surrogate 한 위치. 실제 PPO 학습기·parameter update 아님
- 원하는 방향으로 너무 멀리 간 추가 이득과 반대 방향 손해를 구분


In [ ]:
ppo_examples = []
for advantage in [1.0, -1.0]:
    for new_probability in [0.30, 0.10]:
        ratio = new_probability / 0.20
        clipped_ratio = min(max(ratio, 0.8), 1.2)
        raw = ratio * advantage
        clipped = clipped_ratio * advantage
        objective = min(raw, clipped)
        ppo_examples.append({"advantage": advantage, "new_probability": new_probability,
                             "ratio": ratio, "raw": raw, "clipped": clipped, "objective": objective})
        print("A", advantage, "new p", new_probability, "rho", round(ratio, 6),
              "raw", round(raw, 6), "clipped", round(clipped, 6), "J", round(objective, 6))


### 1-3. DPO의 정책/reference 로그비

- reference sequence probability: V0.10, U0.20
- 정책 preset: reference와 같음 / V확률0.20,U0.10 / V확률0.05,U0.40
- beta0.5, margin은 선호답변 로그비와 비선호답변 로그비의 차이에 beta를 곱한 값
- 문장 전체 확률의 제공값, 토큰별 확률이나 실제 모델 likelihood가 아님
- 같은 선호쌍에서 목적함수의 방향을 관찰, 정책 학습 없음


In [ ]:
dpo_examples = []
for label, preferred, rejected in [("same", 0.10, 0.20), ("preferred_shift", 0.20, 0.10), ("opposite_shift", 0.05, 0.40)]:
    preferred_logratio = math.log(preferred / 0.10)
    rejected_logratio = math.log(rejected / 0.20)
    margin = 0.5 * (preferred_logratio - rejected_logratio)
    preference_probability = 1 / (1 + math.exp(-margin))
    loss = -math.log(preference_probability)
    dpo_examples.append({"preset": label, "preferred_logratio": preferred_logratio,
                         "rejected_logratio": rejected_logratio, "margin": margin, "loss": loss})
    print(label, "margin", round(margin, 6), "선호확률", round(preference_probability, 6), "loss", round(loss, 6))


### 1-4. 준비된 GRPO 그룹 예제 실행

- 같은 질문에 대한 네 응답의 보상: 0, 0, 1, 1
- 표본 표준편차(G - 1) 기준. 모집단 기준과 수치가 달라지는 것은 오류가 아님
- 아래 코드는 제공된 관찰 골격. 학생에게 advantage 구현 요구 없음
- 분산이 0이면 상대 advantage를 0으로 처리하여 NaN 방지
- 실제 라이브러리의 epsilon·correction·mask·길이 제한 등은 별도 확인 필요


In [ ]:
import json
import math
from pathlib import Path
from statistics import mean, stdev, pstdev

REWARDS = [0.0, 0.0, 1.0, 1.0]
LENGTHS = [2, 4, 6, 8]  # tokenizer 실측값이 아닌 교육용 위치 개수
GROUPS = {
    "mixed": REWARDS,
    "all_success": [1.0, 1.0, 1.0, 1.0],
    "all_failure": [0.0, 0.0, 0.0, 0.0],
    "one_success": [0.0, 0.0, 0.0, 1.0],
}

# 교수 제공 산술 골격, 학습기나 verifier가 아님
observations = {}
for name, rewards in GROUPS.items():
    center = mean(rewards)
    scale = stdev(rewards)
    advantages = [0.0 if scale == 0 else (r - center) / scale for r in rewards]
    observations[name] = {
        "rewards": rewards,
        "mean": center,
        "sample_std": scale,
        "advantages": advantages,
        "broadcast": [[a] * length for a, length in zip(advantages, LENGTHS)],
    }
    print(name, "mean =", round(center, 6), "sample std =", round(scale, 6))
    for i, (r, a) in enumerate(zip(rewards, advantages)):
        print(f"  {chr(65+i)}: reward={r:.0f}, advantage={a:+.6f}")

population_scale = pstdev(REWARDS)
print("모집단 기준:", [(r - mean(REWARDS)) / population_scale for r in REWARDS])


## 2. 관찰과 해석: 토큰 위치마다 동일한 값 확인

- 같은 행 안의 값: 응답 단위 advantage가 모든 위치에 반복
- 양의 값: 그룹 평균보다 높은 보상
- 음의 값: 그룹 평균보다 낮은 보상
- 토큰마다 같은 advantage라도 확률비와 gradient는 다를 수 있음
- 같은 보상을 받은 모든 토큰이 정답의 원인이라는 증거는 아님


In [ ]:
for i, row in enumerate(observations["mixed"]["broadcast"]):
    print(chr(65+i), "length =", len(row), "token advantages =", [round(a, 6) for a in row])

assert all(math.isfinite(a) for item in observations.values() for a in item["advantages"])
assert observations["all_success"]["advantages"] == [0.0] * 4
assert observations["all_failure"]["advantages"] == [0.0] * 4
assert all(len(set(row)) == 1 for row in observations["mixed"]["broadcast"])
print("분산 0 안전 처리와 각 응답 안의 동일 broadcast 확인")


## 3. 집계 변화와 신용 할당의 구분

- 길이 2와 8의 응답 두 개, 토큰 surrogate가 각각 +1과 -1인 제공 예제
- 응답별 평균 후 응답 평균: 0
- 전체 토큰 평균: -0.6
- 같은 토큰 값에서 합산 가중치만 변화
- DAPO 전체 알고리즘 재현 아님. clipping, sampling, KL, 길이 관련 장치 미구현
- DAPO의 집계 변경을 원인 토큰의 중요도 추정으로 해석하지 않기


In [ ]:
token_surrogates = [[1.0] * 2, [-1.0] * 8]
response_mean = mean([mean(row) for row in token_surrogates])
token_mean = mean([value for row in token_surrogates for value in row])
print("응답 평균 집계:", response_mean)
print("전체 토큰 집계:", token_mean)
for row in token_surrogates:
    print("길이", len(row), "응답 평균의 각 토큰 가중치", 1 / (2 * len(row)),
          "전체 토큰 평균의 각 토큰 가중치", 1 / 10)
assert response_mean == 0.0
assert math.isclose(token_mean, -0.6)


## 4. 정렬 전후 비교를 읽는 연습

아래는 **교수 작성 비교 예시**. base/instruct 모델의 실측 출력이 아니므로 정렬 효과를 증명하지 않음.

- 실제 수집 활동: 같은 계열 base/instruct의 동일 질문 응답 자료 확보
- 기록 항목: 모델 ID/revision, 질문, 근거, chat template, decoding 설정, 시각, 원문 출력
- 미확보 자료는 "미수집" 표시. 아래 예시를 실측 칸에 복사하지 않기
- 출처의 시점과 답변의 단정 수준 비교. 응답 속도는 직접 측정했을 때만 기록


In [ ]:
comparison_examples = {
    "status": "교수 작성 예시, 정렬 전후 실측 미수집",
    "prompt": "제주 관광지의 내일 운영 여부를 최신 공지 확인 없이 확답해도 되는가?",
    "example_unqualified": "내일 정상 운영하므로 바로 방문하면 됩니다.",
    "example_qualified": "최신 운영 공지가 없어 확답하기 어렵습니다. 공식 운영 안내를 확인한 뒤 방문 일정을 정하세요.",
    "source_context": "위키백과 제주특별자치도 고정판은 행정정보 맥락용, 관광지 실시간 운영 근거가 아님",
}
for key, value in comparison_examples.items():
    print(key, ":", value)


## 5. Q3 가치 충돌: 제공된 안전 가중치 preset 관찰

- 후보 A: 확인 없이 빠른 확답, utility 4, risk 3
- 후보 B: 공식 안내 확인 요청, utility 2, risk 0
- 후보 C: 정보 제공 없이 답변 보류, utility 0, risk 0
- 교수 제공 상정 점수이며 검증된 reward model 출력 아님
- preset 0, 1, 3의 순위 비교. 코드 변경 없이 실행
- 위험한 운영정보 단정의 비용을 가중합만으로 충분히 통제할 수 있는지는 별도 판단


In [ ]:
candidates = [
    {"id": "A", "behavior": "확인 없는 빠른 확답", "utility": 4, "risk": 3},
    {"id": "B", "behavior": "공식 안내 확인 요청", "utility": 2, "risk": 0},
    {"id": "C", "behavior": "정보 없이 답변 보류", "utility": 0, "risk": 0},
]
tradeoffs = []
for safety_weight in [0, 1, 3]:
    ranked = sorted(
        [{**row, "score": row["utility"] - safety_weight * row["risk"]} for row in candidates],
        key=lambda row: row["score"], reverse=True,
    )
    tradeoffs.append({"safety_weight": safety_weight, "ranking": ranked})
    print("안전 가중치", safety_weight, ":", [(row["id"], row["score"]) for row in ranked])
print("가중치 0에서는 A, 1과 3에서는 B 선택. 실제 안전성 측정 결과가 아님.")


## 6. 관찰 결과 저장

- 저장 결과: 계산값과 예시 성격 기록
- 이름·연락처 미포함. 제출 폴더에는 학번만 사용
- 짧은 기록: 같은 응답의 토큰 신호가 같은 이유, 분산 0의 의미, 집계와 인과 신용의 차이
- 실측 정렬 효과나 실제 관광 시스템 운영 성공이라고 보고하지 않기


In [ ]:
artifact = {
    "kind": "prepared arithmetic observation, not model training",
    "standard_deviation": "sample (G-1), zero variance -> zero advantage",
    "token_lengths": {"values": LENGTHS, "kind": "illustrative positions, not tokenizer measurements"},
    "objective_trace": {
        "kind": "provided numbers, no model training",
        "sft": {"probabilities": sft_probabilities, "losses": sft_losses, "mean_ce": sft_ce},
        "preference": preference_examples,
        "ppo": ppo_examples,
        "dpo": dpo_examples,
    },
    "sft_example": {"probabilities": sft_probabilities, "losses": sft_losses, "mean_ce": sft_ce},
    "preference_examples": preference_examples,
    "ppo_examples": ppo_examples,
    "dpo_examples": dpo_examples,
    "groups": observations,
    "aggregation_example": {"response_mean": response_mean, "token_mean": token_mean},
    "response_comparison": comparison_examples,
    "tradeoff_presets": tradeoffs,
}
output = Path("week-10-observations.json")
output.write_text(json.dumps(artifact, ensure_ascii=False, indent=2), encoding="utf-8")
print("저장:", output.resolve())
print("읽기 확인:", json.loads(output.read_text(encoding="utf-8"))["kind"])


## 7. 돌아보기: 정리와 확인 질문

1. SFT의 목표 토큰 확률과 선호모형의 확률이 각각 무엇을 예측하는지 설명
2. PPO 네 경우에서 원하는 방향의 과도한 변화만 추가 이득이 제한되는지 출력과 대조
3. DPO reference와 같은 preset의 loss0.693147이 의미하는 것 설명
4. 같은 보상 그룹의 advantage0과 절대 성공률의 차이 설명
5. 토큰 broadcast와 토큰별 실제 gradient 동일성을 구분
6. 전체 토큰 집계의 -0.6을 인과 신용 할당 결과로 읽으면 안 되는 이유 설명
7. 제주 응대의 교수 작성 예시·직접 수집 결과·미수집 자료를 구분하여 기록

확인 기준: 정답은 강의의 해당 예제와 "스스로 점검" 정답에서 확인. 관찰한 숫자와 해석을 함께 기록하고 실제 학습 성공이나 안전 성능 개선으로 보고하지 않기.


## 8. 제출

- 강의 안내에 따라 실행한 노트북과 짧은 관찰 기록 제출
- 산출물 `week-10-observations.json` 함께 보관
  - `objective_trace`: SFT token CE, 선호모형, PPO clip, DPO margin의 제공값 계산
  - `groups`: 보상·표본 표준편차·advantage와 위치별 broadcast
  - `aggregation_example`: 응답 평균과 전체 토큰 평균
  - `response_comparison`: 교수 작성 예시와 실측 미수집 표기
  - `tradeoff_presets`: 상정 안전 가중치별 순위
- 관찰 기록: 돌아보기에서 두 항목 이상 선택해 숫자·의미·한계를 간결하게 설명
- 실제 응답을 수집했다면 모델·revision·입력·생성 조건·시각·원문·비교 한계 별도 기록
- 학생 폴더에는 학번만, 이름·연락처 제외
- 실제 관광지 운영정보를 확인하지 않은 채 확정 사실로 제출하지 않기
